In [ ]:
# Install required dependencies
print("Installing required dependencies...")

!sudo apt-get install -y libusb-1.0-0-dev

# Clone the MuJoCo repo for ALOHA
!git clone https://github.com/google-deepmind/mujoco_menagerie.git

# Clone the Wiki-GRx-Models repo for GR1 and copy the urdf file to temp folder
!git clone https://github.com/FFTAI/Wiki-GRx-Models.git
!cp -r Wiki-GRx-Models/GRX/GR1/gr1t2 gr1_assets

# !pip install --ignore-installed blinker
!pip install -q "z3-solver==4.15.4.0" "genesis-world==0.3.11" "rerun-sdk[all]==0.28.2" "imageio[ffmpeg]"
!pip uninstall -y numpy numba flash-attn
!pip install -q "numpy==2.2.0" "numba>=0.61.2"
!pip install -q "transformers<5.0.0" "lerobot[intelrealsense,dynamixel]==0.4.3"
!pip install -q wandb
!pip install -U flash-attn --no-build-isolation

print("Dependencies installed, restarting!")
exit()

Installing required dependencies...
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libusb-1.0-doc
The following NEW packages will be installed:
  libusb-1.0-0-dev libusb-1.0-doc
0 upgraded, 2 newly installed, 0 to remove and 42 not upgraded.
Need to get 259 kB of archives.
After this operation, 1,940 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libusb-1.0-0-dev amd64 2:1.0.25-1ubuntu2 [76.3 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libusb-1.0-doc all 2:1.0.25-1ubuntu2 [183 kB]
Fetched 259 kB in 2s (142 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize fronte

In [ ]:
import os

patch_code = """
from transformers import PreTrainedModel
_orig_check = PreTrainedModel._check_and_adjust_attn_implementation

def patched_check_attn(self, attn_implementation, is_init_check):
    if attn_implementation == 'flash_attention_2':
        return 'sdpa'
    return _orig_check(self, attn_implementation, is_init_check)

PreTrainedModel._check_and_adjust_attn_implementation = patched_check_attn
"""

# The traceback shows lerobot-train points here:
target_file = "/usr/local/lib/python3.12/dist-packages/lerobot/scripts/lerobot_train.py"

with open(target_file, "r") as f:
    content = f.read()

# Only patch if we haven't already
if "patched_check_attn" not in content:
    with open(target_file, "w") as f:
        f.write(patch_code + "\n" + content)
    print("Successfully patched lerobot_train.py to disable Flash Attention!")
else:
    print("Script was already patched.")

Successfully patched lerobot_train.py to disable Flash Attention!


## Dummy Data

In [ ]:
!ls /content/drive/MyDrive/Masters_Project

generative_models_playground.ipynb  gr1_teleop_demo
Genesis_Rerun_Demo.ipynb	    LeRobot_GR00T_Basic.ipynb
GR00T_N1_BC.ipynb		    training-smolvla.ipynb
GR00T_N1_E2E.ipynb


In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.tune_diffusion_model=false \
    --policy.push_to_hub=false \
    --dataset.repo_id=local/gr1_teleop_demo \
    --dataset.root=/content/drive/MyDrive/Masters_Project/gr1_teleop_demo \
    --batch_size=1 \
    --steps=10 \
    --save_checkpoint=true \
    --wandb.enable=false \
    --save_freq=10 \
    --output_dir=./outputs/

2026-03-10 17:06:28.053839: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-10 17:06:28.072837: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773162388.095950   13281 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773162388.102791   13281 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773162388.120198   13281 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!mv outputs dummy_outputs
!mv dummy_outputs drive/MyDrive/Masters_Project/dummy_outputs

In [ ]:
!rm -rf outputs

In [ ]:
!du -sh outputs/

11G	outputs/


In [ ]:
!tar -cf /content/dummy_checkpoint.tar -C /content/outputs checkpoints

In [ ]:
!du -sh dummy_checkpoint.tar

11G	dummy_checkpoint.tar


In [ ]:
!mv /content/dummy_checkpoint.tar /content/drive/MyDrive/Masters_Project/

In [ ]:
!ls /content/drive/MyDrive/Masters_Project/

dummy_checkpoint.tar		    GR00T_N1_E2E.ipynb
generative_models_playground.ipynb  gr1_teleop_demo
Genesis_Rerun_Demo.ipynb	    LeRobot_GR00T_Basic.ipynb
GR00T_N1_BC.ipynb		    training-smolvla.ipynb


## Pickup Data

In [ ]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_pickup
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_pickup
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_pickup

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vedpat3 (vedpat3-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `robot` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `robot`


In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --dataset.repo_id=vedpatwardhan/gr1_pickup \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=10000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=5e-6 \
    --batch_size=8 \
    --steps=5000 \
    --save_freq=1000 \
    --output_dir=/content/drive/MyDrive/Masters_Project/gr1_pickup/run1 \
    --wandb.enable=true \
    --wandb.project=gr1-pickup

2026-03-29 12:14:49.843514: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-29 12:14:49.861471: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774786489.883187   12404 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774786489.889797   12404 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774786489.906803   12404 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

In [ ]:
while True: pass

KeyboardInterrupt: 

In [ ]:
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!ls drive

MyDrive


In [ ]:
%%writefile sample.py

from huggingface_hub import HfApi

# 1. Configuration
local_model_path = "/content/drive/MyDrive/Masters_Project/gr1_pickup/run1/checkpoints/last/pretrained_model"
your_hf_username = "vedpatwardhan"  # Change this if your username is different
repo_name = "GR00T-N1.5-finetuned-pickup"
repo_id = f"{your_hf_username}/{repo_name}"

api = HfApi()

# 2. Create the repository on your profile (if it doesn't exist)
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# 3. Upload the folder from Google Drive
print(f"Uploading model to https://huggingface.co/{repo_id}...")
api.upload_folder(
    folder_path=local_model_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Finetuned GR00T-N1.5 on pickup task (Step 5000)"
)
print("✅ Upload Complete!")


Writing sample.py


In [ ]:
!python sample.py

Uploading model to https://huggingface.co/vedpatwardhan/GR00T-N1.5-finetuned-pickup...
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...d_model/model.safetensors:   1% 56.0M/6.96G [00:00<?, ?B/s]

Processing Files (0 / 1)      :   1% 56.0M/6.96G [00:03<07:38, 15.1MB/s, 18.7MB/s  ]

Processing Files (0 / 1)      :   1% 87.9M/6.96G [00:03<04:30, 25.4MB/s, 27.5MB/s  ]

Processing Files (0 / 1)      :   2% 128M/6.96G [00:04<02:44, 41.6MB/s, 37.6MB/s  ] 

Processing Files (0 / 1)      :   2% 163M/6.96G [00:04<01:59, 57.0MB/s, 45.2MB/s  ]

Processing Files (0 / 1)      :   3% 195M/6.96G [00:04<01:34, 71.7MB/s, 51.2MB/s  ]

Processing Files (0 / 1)      :   3% 235M/6.96G [00:04<01:11, 93.5MB/s, 58.7MB/s  ]

Processing Files (0 / 1)      :   4% 267M/6.96G [00:04<01:02, 107MB/s, 63.5MB/s  ] 

Processing Files (0 / 1)      :   4% 307M/6.96G [00:05<00:52, 127MB/s, 69.7MB/s  ]

Processing Files (

In [ ]:
!ls /content/drive/MyDrive/Masters_Project/gr1_pickup/run1/checkpoints/last/

pretrained_model  training_state


In [ ]:
!find /content/drive/MyDrive/Masters_Project/gr1_pickup/run1 -name "metadata.json"

In [ ]:
# This will show the entire directory structure of your training results
!find /content/drive/MyDrive/Masters_Project/gr1_pickup/run1 -maxdepth 5

/content/drive/MyDrive/Masters_Project/gr1_pickup/run1
/content/drive/MyDrive/Masters_Project/gr1_pickup/run1/wandb
/content/drive/MyDrive/Masters_Project/gr1_pickup/run1/wandb/run-20260329_121503-fdzlo5eg
/content/drive/MyDrive/Masters_Project/gr1_pickup/run1/wandb/run-20260329_121503-fdzlo5eg/logs
/content/drive/MyDrive/Masters_Project/gr1_pickup/run1/wandb/run-20260329_121503-fdzlo5eg/logs/debug.log
/content/drive/MyDrive/Masters_Project/gr1_pickup/run1/wandb/run-20260329_121503-fdzlo5eg/logs/debug-internal.log
/content/drive/MyDrive/Masters_Project/gr1_pickup/run1/wandb/run-20260329_121503-fdzlo5eg/logs/debug-core.log
/content/drive/MyDrive/Masters_Project/gr1_pickup/run1/wandb/run-20260329_121503-fdzlo5eg/files
/content/drive/MyDrive/Masters_Project/gr1_pickup/run1/wandb/run-20260329_121503-fdzlo5eg/files/wandb-metadata.json
/content/drive/MyDrive/Masters_Project/gr1_pickup/run1/wandb/run-20260329_121503-fdzlo5eg/files/requirements.txt
/content/drive/MyDrive/Masters_Project/gr1_pi

In [ ]:
%%writefile sample.py

import json
from huggingface_hub import hf_hub_download, HfApi

# --- Configuration ---
repo_id = "vedpatwardhan/GR00T-N1.5-finetuned-pickup"
dataset_repo = "vedpatwardhan/gr1_pickup"
# Using the final step checkpoint
local_ckpt_dir = "/content/drive/MyDrive/Masters_Project/gr1_pickup/run1/checkpoints/last/pretrained_model"

print("🔍 Step 1: Downloading stats from Hub...")
# 1. Download the source of truth stats from your fixed dataset
stats_path = hf_hub_download(repo_id=dataset_repo, filename="meta/stats.json", repo_type="dataset")
with open(stats_path, 'r') as f:
    stats = json.load(f)

print("✍️ Step 2: Constructing metadata...")
# 2. Re-create the deployment metadata
metadata = {
    "policy_type": "groot",
    "fps": 30,
    "video_backend": "torchcodec",
    "stats": stats,
    "input_features": {
        "observation.images.world_center": {"shape": [3, 224, 224], "type": "video"},
        "observation.images.world_left": {"shape": [3, 224, 224], "type": "video"},
        "observation.images.world_right": {"shape": [3, 224, 224], "type": "video"},
        "observation.images.world_top": {"shape": [3, 224, 224], "type": "video"},
        "observation.images.world_wrist": {"shape": [3, 224, 224], "type": "video"},
        "observation.state": {"shape": [64], "type": "vector"}
    },
    "output_features": {
        "action": {"shape": [32], "type": "vector"}
    }
}

# 3. Save it to your Google Drive for persistence
meta_save_path = f"{local_ckpt_dir}/metadata.json"
with open(meta_save_path, 'w') as f:
    json.dump(metadata, f, indent=4)
print(f"📄 metadata.json saved to Drive at: {meta_save_path}")

print("☁️ Step 3: Uploading to Hugging Face...")
# 4. Push just this one file to complete your repo
api = HfApi()
api.upload_file(
    path_or_fileobj=meta_save_path,
    path_in_repo="metadata.json",
    repo_id=repo_id,
    repo_type="model",
    commit_message="Manually inject missing metadata.json after initial upload crash"
)

print(f"\n✅ SUCCESS! Your repo is now complete: https://huggingface.co/{repo_id}")


Overwriting sample.py


In [ ]:
!python sample.py

🔍 Step 1: Downloading stats from Hub...
stats.json: 5.53kB [00:00, 18.3MB/s]
✍️ Step 2: Constructing metadata...
📄 metadata.json saved to Drive at: /content/drive/MyDrive/Masters_Project/gr1_pickup/run1/checkpoints/last/pretrained_model/metadata.json
☁️ Step 3: Uploading to Hugging Face...

✅ SUCCESS! Your repo is now complete: https://huggingface.co/vedpatwardhan/GR00T-N1.5-finetuned-pickup


## Pickup Data Large

In [ ]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_pickup_large
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_pickup_large
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_pickup_large

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vedpat3 (vedpat3-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `robot` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `robot`


In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.embodiment_tag=gr1 \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'MEAN_STD'}" \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_large \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=5e-6 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --output_dir=/content/drive/MyDrive/Masters_Project/gr1_pickup_large/run4 \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-large \
    --wandb.disable_artifact=true

2026-04-04 14:16:05.361149: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-04 14:16:05.379028: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775312165.400935    8353 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775312165.407678    8353 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775312165.424360    8353 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.pretrained_path=/content/drive/MyDrive/Masters_Project/gr1_pickup_large/run4/checkpoints/015000/pretrained_model \
    --policy.embodiment_tag=gr1 \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'MEAN_STD'}" \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_large \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-6 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=0 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-6 \
    --scheduler.decay_lr=1e-7 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --output_dir=/content/drive/MyDrive/Masters_Project/gr1_pickup_large/run5 \
    --resume=false \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-large \
    --wandb.disable_artifact=true

2026-04-05 10:28:12.440016: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-05 10:28:12.457460: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775384892.478790   25055 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775384892.485320   25055 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775384892.501755   25055 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!ls drive/MyDrive/Masters_Project/gr1_pickup_large/run5/checkpoints/015000

pretrained_model  training_state


In [ ]:
%%writefile sample.py

from huggingface_hub import HfApi

# 1. Configuration
local_model_path = "/content/drive/MyDrive/Masters_Project/gr1_pickup_large/run5/checkpoints/015000/pretrained_model"
your_hf_username = "vedpatwardhan"  # Change this if your username is different
repo_name = "GR00T-N1.5-finetuned-pickup-large"
repo_id = f"{your_hf_username}/{repo_name}"

api = HfApi()

# 2. Create the repository on your profile (if it doesn't exist)
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# 3. Upload the folder from Google Drive
print(f"Uploading model to https://huggingface.co/{repo_id}...")
api.upload_folder(
    folder_path=local_model_path,
    repo_id=repo_id,
    repo_type="model",
    commit_message="Finetuned GR00T-N1.5 on pickup task (Step 5000)"
)
print("✅ Upload Complete!")

Writing sample.py


In [ ]:
!python sample.py

Uploading model to https://huggingface.co/vedpatwardhan/GR00T-N1.5-finetuned-pickup-large...
Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...ack_inputs_v3.safetensors: 100% 15.9k/15.9k [00:00<?, ?B/s]


  ...nnormalize_v1.safetensors: 100% 15.9k/15.9k [00:00<?, ?B/s]

  ...ack_inputs_v3.safetensors: 100% 15.9k/15.9k [00:00<?, ?B/s]


Processing Files (2 / 2)      :   0% 31.9k/6.96G [00:00<40:38:27, 47.6kB/s, 79.7kB/s  ]

  ...ack_inputs_v3.safetensors: 100% 15.9k/15.9k [00:00<?, ?B/s]


  ...nnormalize_v1.safetensors: 100% 15.9k/15.9k [00:00<?, ?B/s]

  ...ack_inputs_v3.safetensors: 100% 15.9k/15.9k [00:00<?, ?B/s]


  ...nnormalize_v1.safetensors: 100% 15.9k/15.9k [00:00<?, ?B/s]



  ...d_model/model.safetensors:   1% 48.0M/6.96G [00:00<?, ?B/s]

  ...ack_inputs_v3.safetensors: 100% 15.9k/15.9k [00:00<?, ?B/s]


  ...nnormalize_v1.safetensors: 100% 15.9k/15.9k [00:00<?, ?B/s]



Pr

## Pickup Data Compact

In [ ]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_pickup_compact_h264
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_pickup_compact_h264
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_pickup_compact_h264

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vedpat3 (vedpat3-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `robot` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `robot`


In [ ]:
!cp -r /content/drive/MyDrive/Masters_Project/gr1_pickup_compact_h264 gr1_pickup_compact_h264

In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.embodiment_tag=gr1 \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'MEAN_STD'}" \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_compact_h264 \
    --dataset.video_backend=torchcodec \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=1e-7 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --num_workers=8 \
    --output_dir=/content/drive/MyDrive/Masters_Project/gr1_pickup_compact/run6 \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-compact \
    --wandb.disable_artifact=true

2026-04-14 06:06:35.661648: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-14 06:06:35.678851: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776146795.700942   12838 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776146795.707520   12838 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776146795.723937   12838 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

## Pickup Data Final

In [ ]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_pickup_final
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_pickup_final
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_pickup_final

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vedpat3 (vedpat3-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `robot` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `robot`


In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.embodiment_tag=gr1 \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'IDENTITY'}" \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_final \
    --dataset.video_backend=torchcodec \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=1e-7 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --num_workers=8 \
    --output_dir=/content/drive/MyDrive/Masters_Project/gr1_pickup_final/run1 \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-final \
    --wandb.disable_artifact=true

Streaming output truncated to the last 5000 lines.
            'lora_full_model': False,
            'lora_rank': 0,
            'max_action_dim': 32,
            'max_state_dim': 64,
            'max_steps': 10000,
            'n_action_steps': 16,
            'n_obs_steps': 1,
            'normalization_mapping': {'ACTION': <NormalizationMode.IDENTITY: 'IDENTITY'>,
                                      'STATE': <NormalizationMode.IDENTITY: 'IDENTITY'>},
            'optimizer_betas': [0.95, 0.999],
            'optimizer_eps': 1e-08,
            'optimizer_lr': 0.0001,
            'optimizer_weight_decay': 1e-05,
            'output_dir': './tmp/gr00t',
            'output_features': {},
            'pretrained_path': None,
            'private': None,
            'push_to_hub': True,
            'repo_id': 'nvidia/GR00T-N1.5-3B',
            'report_to': 'wandb',
            'resume': False,
            'save_steps': 1000,
            'tags': None,
            'tokenizer_assets_repo

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

## Pickup Data Compressed

In [ ]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_pickup_compressed
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_pickup_compressed
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_pickup_compressed

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vedpat3 (vedpat3-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `robot` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `robot`


In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.embodiment_tag=gr1 \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'IDENTITY'}" \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_compressed \
    --dataset.video_backend=torchcodec \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=1e-7 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --num_workers=8 \
    --output_dir=/content/drive/MyDrive/Masters_Project/gr1_pickup_compressed/run3 \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-final \
    --wandb.disable_artifact=true

2026-04-18 11:54:10.275021: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-18 11:54:10.292957: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776513250.314712   34999 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776513250.321319   34999 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776513250.338309   34999 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

## Pickup Data Reward

In [ ]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_pickup_reward
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_pickup_reward
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_pickup_reward

In [ ]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: vedpat3 (vedpat3-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    A token is already saved on your machine. Run `hf auth whoami` to get more information or `hf auth logout` if you want to log out.
    Setting a new token will erase the existing one.
    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The tok

In [ ]:
"""
Patching script to enable Reward-Augmented Behavioral Cloning (RA-BC) for the GR-1 Groot policy.
This script resolves:
1. TypeError: GrootPolicy.forward() got an unexpected keyword argument 'reduction'
2. RuntimeError: a Tensor with 8 elements cannot be converted to Scalar (during logging)
3. Implementation of per-sample loss in the FlowMatchingActionHead.
"""

import os
import lerobot

def patch_file(path, search_text, replace_text):
    if not os.path.exists(path):
        print(f"⚠️ Warning: File not found: {path}")
        return False

    with open(path, "r") as f:
        content = f.read()

    if replace_text in content:
        print(f"✅ Already patched: {os.path.basename(path)}")
        return True

    if search_text not in content:
        print(f"❌ Error: Search text not found in {os.path.basename(path)}")
        # Try a more flexible search if it's the specific loss block
        return False

    new_content = content.replace(search_text, replace_text)
    with open(path, "w") as f:
        f.write(new_content)
    print(f"🚀 Successfully patched: {os.path.basename(path)}")
    return True

def main():
    lib_path = os.path.dirname(lerobot.__file__)
    print(f"🔍 Found LeRobot at: {lib_path}")

    # 1. Patch modeling_groot.py: Signature + RuntimeError FIX
    p1 = os.path.join(lib_path, "policies/groot/modeling_groot.py")
    # Add reduction to signature
    patch_file(p1,
        "def forward(self, batch: dict[str, Tensor]) ->",
        "def forward(self, batch: dict[str, Tensor], reduction: str = 'mean') ->")
    # Propagate reduction
    patch_file(p1,
        "outputs = self._groot_model.forward(groot_inputs)",
        "outputs = self._groot_model.forward(groot_inputs, reduction=reduction)")
    # FIX RuntimeError (8 elements cannot be converted to scalar)
    patch_file(p1,
        'loss_dict = {"loss": loss.item()}',
        'loss_dict = {"loss": loss.mean().item()}')

    # 2. Patch groot_n1.py: Propagation Fix
    p2 = os.path.join(lib_path, "policies/groot/groot_n1.py")
    patch_file(p2,
        "forward(\n        self,\n        inputs: dict,\n    ) ->",
        "forward(\n        self,\n        inputs: dict,\n        reduction: str = 'mean',\n    ) ->")
    patch_file(p2,
        "action_head_outputs = self.action_head(backbone_outputs, action_inputs)",
        "action_head_outputs = self.action_head(backbone_outputs, action_inputs, reduction=reduction)")

    # 3. Patch flow_matching_action_head.py: Weighted Loss Implementation
    p3 = os.path.join(lib_path, "policies/groot/action_head/flow_matching_action_head.py")
    patch_file(p3,
        "forward(self, backbone_output: BatchFeature, action_input: BatchFeature) ->",
        "forward(self, backbone_output: BatchFeature, action_input: BatchFeature, reduction: str = 'mean') ->")

    old_loss = """        # Slice out only the action portion of pred and target.
        action_mask = action_input.action_mask
        loss = F.mse_loss(pred_actions, velocity, reduction="none") * action_mask
        loss = loss.sum() / action_mask.sum()"""

    new_loss = """        # Slice out only the action portion of pred and target.
        action_mask = action_input.action_mask
        if reduction == "none":
            # Return per-sample loss for RA-BC weighting
            loss = F.mse_loss(pred_actions, velocity, reduction="none") * action_mask
            loss = loss.mean(dim=(1, 2))
        else:
            loss = F.mse_loss(pred_actions, velocity, reduction="none") * action_mask
            loss = loss.sum() / action_mask.sum()"""

    patch_file(p3, old_loss, new_loss)

    print("\n✨ RA-BC Patching Complete. You can now run !lerobot-train normally.")

main()

🔍 Found LeRobot at: /usr/local/lib/python3.12/dist-packages/lerobot
🚀 Successfully patched: modeling_groot.py
🚀 Successfully patched: modeling_groot.py
🚀 Successfully patched: modeling_groot.py
🚀 Successfully patched: groot_n1.py
🚀 Successfully patched: groot_n1.py
🚀 Successfully patched: flow_matching_action_head.py
🚀 Successfully patched: flow_matching_action_head.py

✨ RA-BC Patching Complete. You can now run !lerobot-train normally.


In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.embodiment_tag=gr1 \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'IDENTITY'}" \
    --use_rabc=true \
    --rabc_head_mode=sparse \
    --rabc_kappa=0.01 \
    --rabc_progress_path=hf://datasets/vedpatwardhan/gr1_pickup_reward/progress_fixed.parquet \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_reward \
    --dataset.video_backend=torchcodec \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=1e-7 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --num_workers=8 \
    --output_dir=/content/drive/MyDrive/Masters_Project/gr1_pickup_reward/run3 \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-reward \
    --wandb.disable_artifact=true

2026-04-19 16:17:22.874945: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-19 16:17:22.893012: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776615442.915188   22253 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776615442.921968   22253 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776615442.938754   22253 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

## Pickup Data 32

In [ ]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_pickup_32
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_pickup_32
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_pickup_32

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 8f6151fdd76ca85dbd35844ca5b9c53af1c38a39


wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vedpat3 (vedpat3-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `robot` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `robot`


### Naive BC

In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.embodiment_tag=gr1 \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'IDENTITY'}" \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_32 \
    --dataset.video_backend=torchcodec \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=1e-7 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --num_workers=8 \
    --output_dir=/content/drive/MyDrive/Masters_Project/gr1_pickup_32/run1 \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-32 \
    --wandb.disable_artifact=true

Streaming output truncated to the last 5000 lines.



videos/observation.images.world_wrist/ch(…):   0% 0.00/152k [00:01<?, ?B/s]







videos/observation.images.world_wrist/ch(…):   0% 0.00/164k [00:00<?, ?B/s]



videos/observation.images.world_wrist/ch(…): 100% 179k/179k [00:02<00:00, 89.2kB/s]





videos/observation.images.world_wrist/ch(…):   0% 0.00/164k [00:00<?, ?B/s]


videos/observation.images.world_wrist/ch(…):   0% 0.00/171k [00:01<?, ?B/s]

videos/observation.images.world_wrist/ch(…):   0% 0.00/169k [00:01<?, ?B/s]
videos/observation.images.world_wrist/ch(…): 100% 154k/154k [00:02<00:00, 69.7kB/s]
Fetching 1406 files:  89% 1255/1406 [03:40<00:48,  3.09it/s]






videos/observation.images.world_wrist/ch(…): 100% 152k/152k [00:01<00:00, 125kB/s]








videos/observation.images.world_wrist/ch(…): 100% 164k/164k [00:01<00:00, 162kB/s]





videos/observation.images.world_wrist/ch(…):   0% 0.00/164k [00:00<?, ?B/s]


videos/observation.images.world_wrist/ch(…): 100% 171k/1

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

### RABC

In [ ]:
"""
Patching script to enable Reward-Augmented Behavioral Cloning (RA-BC) for the GR-1 Groot policy.
This script resolves:
1. TypeError: GrootPolicy.forward() got an unexpected keyword argument 'reduction'
2. RuntimeError: a Tensor with 8 elements cannot be converted to Scalar (during logging)
3. Implementation of per-sample loss in the FlowMatchingActionHead.
"""

import os
import lerobot

def patch_file(path, search_text, replace_text):
    if not os.path.exists(path):
        print(f"⚠️ Warning: File not found: {path}")
        return False

    with open(path, "r") as f:
        content = f.read()

    if replace_text in content:
        print(f"✅ Already patched: {os.path.basename(path)}")
        return True

    if search_text not in content:
        print(f"❌ Error: Search text not found in {os.path.basename(path)}")
        # Try a more flexible search if it's the specific loss block
        return False

    new_content = content.replace(search_text, replace_text)
    with open(path, "w") as f:
        f.write(new_content)
    print(f"🚀 Successfully patched: {os.path.basename(path)}")
    return True

def main():
    lib_path = os.path.dirname(lerobot.__file__)
    print(f"🔍 Found LeRobot at: {lib_path}")

    # 1. Patch modeling_groot.py: Signature + RuntimeError FIX
    p1 = os.path.join(lib_path, "policies/groot/modeling_groot.py")
    # Add reduction to signature
    patch_file(p1,
        "def forward(self, batch: dict[str, Tensor]) ->",
        "def forward(self, batch: dict[str, Tensor], reduction: str = 'mean') ->")
    # Propagate reduction
    patch_file(p1,
        "outputs = self._groot_model.forward(groot_inputs)",
        "outputs = self._groot_model.forward(groot_inputs, reduction=reduction)")
    # FIX RuntimeError (8 elements cannot be converted to scalar)
    patch_file(p1,
        'loss_dict = {"loss": loss.item()}',
        'loss_dict = {"loss": loss.mean().item()}')

    # 2. Patch groot_n1.py: Propagation Fix
    p2 = os.path.join(lib_path, "policies/groot/groot_n1.py")
    patch_file(p2,
        "forward(\n        self,\n        inputs: dict,\n    ) ->",
        "forward(\n        self,\n        inputs: dict,\n        reduction: str = 'mean',\n    ) ->")
    patch_file(p2,
        "action_head_outputs = self.action_head(backbone_outputs, action_inputs)",
        "action_head_outputs = self.action_head(backbone_outputs, action_inputs, reduction=reduction)")

    # 3. Patch flow_matching_action_head.py: Weighted Loss Implementation
    p3 = os.path.join(lib_path, "policies/groot/action_head/flow_matching_action_head.py")
    patch_file(p3,
        "forward(self, backbone_output: BatchFeature, action_input: BatchFeature) ->",
        "forward(self, backbone_output: BatchFeature, action_input: BatchFeature, reduction: str = 'mean') ->")

    old_loss = """        # Slice out only the action portion of pred and target.
        action_mask = action_input.action_mask
        loss = F.mse_loss(pred_actions, velocity, reduction="none") * action_mask
        loss = loss.sum() / action_mask.sum()"""

    new_loss = """        # Slice out only the action portion of pred and target.
        action_mask = action_input.action_mask
        if reduction == "none":
            # Return per-sample loss for RA-BC weighting
            loss = F.mse_loss(pred_actions, velocity, reduction="none") * action_mask
            loss = loss.mean(dim=(1, 2))
        else:
            loss = F.mse_loss(pred_actions, velocity, reduction="none") * action_mask
            loss = loss.sum() / action_mask.sum()"""

    patch_file(p3, old_loss, new_loss)

    print("\n✨ RA-BC Patching Complete. You can now run !lerobot-train normally.")

main()

In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.embodiment_tag=gr1 \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'IDENTITY'}" \
    --use_rabc=true \
    --rabc_head_mode=sparse \
    --rabc_kappa=0.01 \
    --rabc_progress_path=hf://datasets/vedpatwardhan/gr1_pickup_32/progress_sparse.parquet \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_32 \
    --dataset.video_backend=torchcodec \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=1e-7 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --num_workers=8 \
    --output_dir=/content/drive/MyDrive/Masters_Project/gr1_pickup_32/run1 \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-32 \
    --wandb.disable_artifact=true

In [ ]:
from google.colab import drive
drive.flush_and_unmount()

## Pickup Data Grasp

In [ ]:
!rm -rf /root/.cache/huggingface/lerobot/vedpatwardhan/gr1_pickup_grasp
!rm -rf /root/.cache/huggingface/hub/datasets--vedpatwardhan--gr1_pickup_grasp
!rm -rf /root/.cache/huggingface/hub/.locks/datasets--vedpatwardhan--gr1_pickup_grasp

In [ ]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vedpat3 (vedpat3-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: fineGrained).
The token `robot` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `robot`


In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.embodiment_tag=gr1 \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'IDENTITY'}" \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_grasp \
    --dataset.video_backend=torchcodec \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=1e-7 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --num_workers=8 \
    --output_dir=/content/drive/MyDrive/Masters_Project/gr1_pickup_grasp/run1 \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-grasp \
    --wandb.disable_artifact=true

2026-04-22 10:54:22.284004: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-22 10:54:22.301504: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776855262.322608   12782 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776855262.328931   12782 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776855262.345296   12782 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
!lerobot-train \
    --policy.type=groot \
    --policy.repo_id=nvidia/GR00T-N1.5-3B \
    --policy.embodiment_tag=new_embodiment \
    --policy.n_action_steps=16 \
    --policy.chunk_size=16 \
    --policy.normalization_mapping="{'ACTION': 'IDENTITY', 'STATE': 'IDENTITY'}" \
    --dataset.repo_id=vedpatwardhan/gr1_pickup_grasp \
    --dataset.video_backend=torchcodec \
    --dataset.use_imagenet_stats=false \
    --optimizer.lr=5e-5 \
    --use_policy_training_preset=false \
    --scheduler.type=cosine_decay_with_warmup \
    --scheduler.num_warmup_steps=500 \
    --scheduler.num_decay_steps=15000 \
    --scheduler.peak_lr=5e-5 \
    --scheduler.decay_lr=1e-7 \
    --batch_size=8 \
    --steps=15000 \
    --save_freq=3000 \
    --num_workers=8 \
    --output_dir=/content/drive/MyDrive/Masters_Project/gr1_pickup_grasp/run3 \
    --wandb.enable=true \
    --wandb.project=gr1-pickup-grasp \
    --wandb.disable_artifact=true

2026-04-23 05:35:53.803789: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-23 05:35:53.821675: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776922553.843292   14008 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776922553.849763   14008 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776922553.866290   14008 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [ ]:
from google.colab import drive
drive.flush_and_unmount()